Importaciones de librerías y constantes

In [5]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import csv
import gc
import os
import time

import cv2
import torch
import numpy as np
from ultralytics import YOLO
from PIL import Image

from scripts.ocr_models import build_model, clean_text, MODEL_NAMES
from scripts.load_rtdetr import load_rtdetr

PROJECT_DIR = os.environ.get("PROJECT_DIR", ".")
VIDEO_PATH = os.path.join(PROJECT_DIR, "videos/C0142.MP4")
PLATE_WEIGHTS = os.path.join(PROJECT_DIR, "best.pt")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "video_results")

OCR_NAME = MODEL_NAMES[4]

OUT_WIDTH = 1280
IMGSZ = 640
SHOW = False
SAVE = True

CLASSES = [0, 2, 3, 5]    # COCO: person, car, motorcycle, bus
VEHICLES = {"car", "bus", "motorcycle"}

MIN_PLATE_CONF = 0.25
TIME_BETWEEN_OCR = 10
MIN_PLATE_CHARS = 4       # lecturas más cortas se descartan por ser ruido
PADDING = 0.05            # margen al recortar la matrícula

USE_YOLO = True

# Detector de matrículas

### Carga del modelo de yolo:

In [ ]:
plates_detector = YOLO("runs/detect/plates_yolo11s-2/weights/best.pt")
plate_detector_name = "YOLO"

USE_YOLO = True

### Carga del modelo de RT-DETR:

In [6]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

plates_detector, postprocessor = load_rtdetr(
                                    model_path="runs/detect/rtdetr_plate_ft_unfrozen/checkpoint.pth",
                                    config_path="scripts/rtdetr_plate.yml",
                                    device=DEVICE
                                )

USE_YOLO = False
plate_detector_name = "RTDETR"

PLATE_IMGSZ = 800

@torch.no_grad()
def detect(bgr, conf=0.6):
    h, w = bgr.shape[:2]
    im = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)).resize((PLATE_IMGSZ, PLATE_IMGSZ), Image.BILINEAR)
    x = torch.from_numpy(np.array(im)).permute(2, 0, 1).float().div(255)[None].to(DEVICE)
    sizes = torch.tensor([[w, h]], dtype=torch.float32, device=DEVICE)   # (w, h) del recorte original
    labels, boxes, scores = postprocessor(plates_detector(x), sizes)
    labels, boxes, scores = labels[0].cpu().numpy(), boxes[0].cpu().numpy(), scores[0].cpu().numpy()
    keep = scores >= conf
    return boxes[keep], scores[keep], labels[keep]

### Funciones auxiliares

In [7]:
def get_best_plate(plate_model, vehicle_crop, is_yolo):
    if is_yolo:
        res = plate_model.predict(vehicle_crop, imgsz=IMGSZ, conf=MIN_PLATE_CONF, verbose=False)[0]
        if len(res.boxes) == 0:
            return None
        
        index = int(res.boxes.conf.argmax())
        px1, py1, px2, py2 = map(int, res.boxes.xyxy[index])

        return px1, py1, px2, py2, float(res.boxes.conf[index])
    else:
        boxes, scores, _ = detect(vehicle_crop, conf=0.6)
        if len(boxes) == 0:
            return None
        
        index = int(np.argmax(scores))
        px1, py1, px2, py2 = boxes[index]

        return px1, py1, px2, py2, float(scores[index])


In [8]:
out_video = os.path.join(OUTPUT_DIR, f"salida_rtdetr_{OCR_NAME}.mp4")
out_csv = os.path.join(OUTPUT_DIR, f"tracking_rtdetr_{OCR_NAME}.csv")

# Modelos nuevos en cada pasada, así el tracker empieza de cero y todos parten igual
vehicle_model = YOLO("yolo11n.pt")
ocr = build_model(OCR_NAME)

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
src_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
src_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

out_h = int(src_h * OUT_WIDTH / src_w)
scale = OUT_WIDTH / src_w          # Para pasar coordenadas originales al vídeo de salida

writer = None
if SAVE:
    writer = cv2.VideoWriter(out_video, cv2.VideoWriter_fourcc(*"mp4v"),
                                fps, (OUT_WIDTH, out_h))

person_ids, vehicle_ids = set(), set()

# Mejor lectura de cada vehículo hasta el momento: track_id -> dict
best_reading = {}
last_ocr_frame = {}

ocr_calls = 0
ocr_time = 0.0

csv_file = open(out_csv, "w", newline="", encoding="utf-8")
csv_writer = csv.writer(csv_file)
csv_writer.writerow(["fotograma", "tipo_objeto", "confianza", "identificador_tracking",
                        "x1", "y1", "x2", "y2", "matrícula_en_su_caso", "confianza",
                        "mx1", "my1", "mx2", "my2", "texto_matricula"])

results = vehicle_model.track(source=VIDEO_PATH, stream=True, persist=True,
                                classes=CLASSES, imgsz=IMGSZ, verbose=False)

t_total = time.perf_counter()
frame_idx = 0
for r in results:
    orig = r.orig_img
    oh, ow = orig.shape[:2]

    frame = cv2.resize(r.plot(), (OUT_WIDTH, out_h))

    for box in r.boxes:
        if box.id is None:
            continue

        name = vehicle_model.names[int(box.cls[0])]
        track_id = int(box.id[0])
        conf = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        x1, y1, x2, y2 = max(x1, 0), max(y1, 0), min(x2, ow), min(y2, oh)

        plate_label, plate_conf = "", ""
        mx1 = my1 = mx2 = my2 = ""
        plate_text = ""

        if name in VEHICLES:
            vehicle_ids.add(track_id)
            vehicle_crop = orig[y1:y2, x1:x2]

            if vehicle_crop.size == 0:
                continue

            found = get_best_plate(plates_detector, vehicle_crop, USE_YOLO)
            
            if found is not None:
                px1, py1, px2, py2, pconf = found

                # Coordenadas de la matrícula respecto al fotograma completo
                mx1, my1, mx2, my2 = x1 + px1, y1 + py1, x1 + px2, y1 + py2
                plate_label, plate_conf = "plate", pconf

                # Colo leemos con OCR de vez en cuando para no ir lentos
                due = frame_idx - last_ocr_frame.get(track_id, -TIME_BETWEEN_OCR) >= TIME_BETWEEN_OCR
                if due:
                    bw, bh = mx2 - mx1, my2 - my1
                    cx1 = max(int(mx1 - bw * PADDING), 0)
                    cy1 = max(int(my1 - bh * PADDING), 0)
                    cx2 = min(int(mx2 + bw * PADDING), ow)
                    cy2 = min(int(my2 + bh * PADDING), oh)
                    plate_crop = orig[cy1:cy2, cx1:cx2]

                    if plate_crop.size > 0:
                        t0 = time.perf_counter()
                        try:
                            text, _ = ocr.read(plate_crop)
                        except Exception as e:
                            text = ""
                            print(f"  fallo de OCR en fotograma {frame_idx}: {e}")
                        ocr_time += time.perf_counter() - t0
                        ocr_calls += 1

                        text = clean_text(text)
                        last_ocr_frame[track_id] = frame_idx

                        # guardamos la lectura si es plausible y la matrícula se vio mejor
                        prev = best_reading.get(track_id)
                        if len(text) >= MIN_PLATE_CHARS and (prev is None or pconf > prev["det_conf"]):
                            best_reading[track_id] = {"text": text, "det_conf": pconf}

                # dibujamos la caja de la matrícula en el vídeo de salida
                cv2.rectangle(frame, (int(mx1 * scale), int(my1 * scale)),
                                (int(mx2 * scale), int(my2 * scale)), (0, 0, 255), 2)

        # texto leído encima del vehículo (si ya tenemos alguno)
        if track_id in best_reading:
            plate_text = best_reading[track_id]["text"]
            tx, ty = int(x1 * scale), int(y2 * scale) + 22
            (tw, th), _ = cv2.getTextSize(plate_text, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)
            cv2.rectangle(frame, (tx, ty - th - 6), (tx + tw + 6, ty + 6), (0, 0, 0), -1)
            cv2.putText(frame, plate_text, (tx + 3, ty),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
                

        elif name == "person":
            person_ids.add(track_id)

        csv_writer.writerow([frame_idx, name, conf, track_id, x1, y1, x2, y2,
                                         plate_label, plate_conf, mx1, my1, mx2, my2, plate_text])

    # contadores y nombre del OCR usado
    cv2.rectangle(frame, (10, 10), (280, 90), (0, 0, 0), -1)
    cv2.putText(frame, f"Vehiculos: {len(vehicle_ids)}", (20, 45),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    cv2.putText(frame, f"Personas: {len(person_ids)}", (20, 78),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(frame, f"OCR: {OCR_NAME}", (OUT_WIDTH - 300, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    if writer:
        writer.write(frame)

    if SHOW:
        cv2.imshow("Tracking", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    frame_idx += 1

csv_file.close()

if writer:
    writer.release()
if SHOW:
    cv2.destroyAllWindows()

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

c:\Users\aserg\anaconda3\envs\ocr_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aserg\.cache\huggingface\hub\models--HuggingFaceTB--SmolVLM-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


chat_template.json:   0%|          | 0.00/516 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

... (5 outputs left out)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/4.49G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

The channel dimension is ambiguous. Got image shape (4, 5, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape (3, 4, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape (4, 3, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape (3, 3, 3). A